In [1]:
import pandas as pd, numpy as np, os
import duckdb

In [2]:
con = duckdb.connect()
for t in ["user", "activity", "transactions", "status"]:
    con.execute(f"CREATE TABLE {t} AS SELECT * FROM read_csv_auto('data/clean/{t}_clean.csv')")

def run(sql):
    print(con.execute(sql).df().to_string(index=False), "\n")

### 1. Calculate total users, active vs churn users, and churn rate.

In [3]:
run("SELECT * FROM status")

 column0  user_id  loan_count  avg_transaction_amount  is_churn churn_date
       0        1           7            4.252810e+06         0        NaT
       1        2           6            3.331625e+06         0        NaT
       2        3           6            3.574123e+06         0        NaT
       3        4           2            2.630618e+06         0        NaT
       4        5           5            4.629785e+06         0        NaT
       5        6           6            3.263206e+06         0        NaT
       6        7           7            3.748043e+06         0        NaT
       7        8           5            3.656612e+06         0        NaT
       8        9           4            3.591476e+06         0        NaT
       9       10           4            3.860386e+06         0        NaT
      10       11           6            2.858893e+06         0        NaT
      11       12           1            5.946899e+06         1 2025-01-08
      12       13        

In [4]:
run("""
SELECT COUNT(*) AS total_users,
    SUM(CASE WHEN is_churn = 0 THEN 1 ELSE 0 END) AS active_users,
    SUM(CASE WHEN is_churn = 1 THEN 1 ELSE 0 END) AS churn_users,
    ROUND(100.0 * SUM(is_churn)/COUNT(*), 2) AS churn_rate

FROM status
""")

 total_users  active_users  churn_users  churn_rate
         250         176.0         74.0        29.6 



### 2. Calculate churn rate by acquisition channel and signup cohort

In [6]:
run("SELECT * FROM user")

 user_id signup_date acquisition_channel  age     city  age_missing
       1  2024-01-01            Referral 33.0  Jakarta            0
       2  2024-01-02           Affiliate 23.0  Bandung            0
       3  2024-01-03             Organic 26.0  Jakarta            0
       4  2024-01-04            Referral 28.0  Bandung            0
       5  2024-01-05            Referral 47.0  Bandung            0
       6  2024-01-06           Affiliate 29.0  Bandung            0
       7  2024-01-07             Organic 53.0  Bandung            0
       8  2024-01-08             Organic 44.0  Bandung            0
       9  2024-01-09            Referral 35.0 Surabaya            0
      10  2024-01-10                 Ads  NaN  Jakarta            1
      11  2024-01-11            Referral 52.0 Surabaya            0
      12  2024-01-12            Referral 44.0  Bandung            0
      13  2024-01-13            Referral 32.0  Jakarta            0
      14  2024-01-14            Referral 22.0  B

In [7]:
# Churn rate by acquisition channel

run("""
SELECT u.acquisition_channel, COUNT(u.user_id) AS total_users, SUM(s.is_churn) AS churn_users,
       ROUND(100.0 * SUM(s.is_churn) / COUNT(*), 2) AS churn_rate_pct
FROM user u JOIN status s USING (user_id)
GROUP BY u.acquisition_channel ORDER BY churn_rate_pct DESC
""")

acquisition_channel  total_users  churn_users  churn_rate_pct
           Referral           67         21.0           31.34
            Organic           59         18.0           30.51
                Ads           56         16.0           28.57
          Affiliate           68         19.0           27.94 



In [8]:
# Churn rate by signup cohort

run("""
SELECT CAST(DATE_TRUNC('month', u.signup_date) AS DATE) AS signup_cohort,
       COUNT(*) AS total_users, SUM(s.is_churn) AS churn_users,
       ROUND(100.0 * SUM(s.is_churn) / COUNT(*), 2) AS churn_rate_pct
FROM user u JOIN status s USING (user_id)
GROUP BY 1 ORDER BY 1
""")

signup_cohort  total_users  churn_users  churn_rate_pct
   2024-01-01           31          7.0           22.58
   2024-02-01           29         10.0           34.48
   2024-03-01           31         12.0           38.71
   2024-04-01           30          8.0           26.67
   2024-05-01           31         12.0           38.71
   2024-06-01           30          6.0           20.00
   2024-07-01           31          8.0           25.81
   2024-08-01           31          9.0           29.03
   2024-09-01            6          2.0           33.33 



### 3.  Identify users who have not logged in for the last 30 days but were previously active.

In [9]:
run("SELECT * FROM activity")

 column0  user_id event_date  session_count feature_used
       0        1 2025-03-26              1      payment
       1        1 2025-01-01              3   loan_apply
       2        1 2025-03-18              4      payment
       3        1 2025-03-06              1  promo_click
       4        1 2025-02-01              2  promo_click
       5        2 2025-02-08              2      payment
       6        2 2025-02-03              3       browse
       7        2 2025-02-19              4   loan_apply
       8        2 2025-02-23              4   loan_apply
       9        2 2025-04-04              3   loan_apply
      10        2 2025-01-17              5   loan_apply
      11        2 2025-01-14              2       browse
      12        2 2025-02-20              4      payment
      13        2 2025-02-07              2      payment
      14        2 2025-03-23              3       browse
      15        2 2025-03-04              2  promo_click
      16        2 2025-02-07   

In [10]:
run("""
WITH ref AS (SELECT MAX(event_date) AS ref_date FROM activity),
last_act AS (
  SELECT user_id, MAX(event_date) AS last_event_date,
         COUNT(*) AS total_events, SUM(session_count) AS total_sessions
  FROM activity GROUP BY user_id)
SELECT l.user_id, l.last_event_date, r.ref_date - l.last_event_date AS days_inactive,
       l.total_events, l.total_sessions, s.is_churn
FROM last_act l CROSS JOIN ref r JOIN status s USING (user_id)
WHERE l.last_event_date < r.ref_date - INTERVAL '30 days'
ORDER BY days_inactive DESC
""")

 user_id last_event_date  days_inactive  total_events  total_sessions  is_churn
      83      2025-02-25             64             9            28.0         1
     147      2025-03-10             51             5            17.0         0
     155      2025-03-11             50             8            31.0         1
     222      2025-03-15             46            14            53.0         1
      43      2025-03-21             40             6            24.0         1
      50      2025-03-21             40            13            38.0         1
     198      2025-03-21             40            10            26.0         1
     188      2025-03-22             39             5            13.0         0
     137      2025-03-25             36             7            27.0         1
       1      2025-03-26             35             5            11.0         0
      14      2025-03-28             33            10            34.0         1
      59      2025-03-29             32 

### 4.  Identify top 3 acquisition channels with best retention and highest churn.


In [11]:
run("""
WITH t AS (
  SELECT u.acquisition_channel AS channel, COUNT(*) AS total_users,
         SUM(s.is_churn) AS churn_users, 1.0 * SUM(s.is_churn) / COUNT(*) AS churn_rate
  FROM user u JOIN status s USING (user_id) GROUP BY u.acquisition_channel)
SELECT channel, total_users, churn_users,
       ROUND(100 * (1 - churn_rate), 2) AS retention_pct, ROUND(100 * churn_rate, 2) AS churn_pct,
       RANK() OVER (ORDER BY churn_rate ASC)  AS retention_rank,
       RANK() OVER (ORDER BY churn_rate DESC) AS churn_rank
FROM t ORDER BY retention_rank
""")

  channel  total_users  churn_users  retention_pct  churn_pct  retention_rank  churn_rank
Affiliate           68         19.0          72.06      27.94               1           4
      Ads           56         16.0          71.43      28.57               2           3
  Organic           59         18.0          69.49      30.51               3           2
 Referral           67         21.0          68.66      31.34               4           1 

